# Beginner Tutorial 9: Machine Learning for Smarter Workflows

## What You Will Learn

- What machine learning is (finding patterns in data)
- Training vs. inference
- How LearnedAdvisor predicts resource needs
- What surrogate models (emulators) are
- Uncertainty and confidence thresholds
- Active learning (choosing what to learn next)

## Prerequisites

- Completed notebooks 01, 03, 06
- `pip install scalable[ml]`

In [1]:
import os
import tempfile

project_dir = tempfile.mkdtemp(prefix="scalable-beginner-09-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

Working directory: /var/folders/8c/787g_pzs7wq7hljnnx3prdzr0000gn/T/scalable-beginner-09-rhfzmo8w


## 💡 Key Concept: What is Machine Learning?

**Machine learning (ML)** = teaching computers to find patterns in data
and make predictions.

**Traditional programming:** Human writes rules → computer follows them
```
IF memory > 8GB THEN allocate 16GB
```

**Machine learning:** Computer finds rules from data
```
Training data: [past runs with memory usage]
ML learns: "scenarios with >1000 nodes need ~12GB"
Prediction: "scenario 47 (1200 nodes) → recommend 16GB"
```

**Analogy:** Traditional = following a recipe. ML = learning to cook from experience.

## 💡 Key Concept: Training vs. Inference

**Training** (slow, done once): Feed data → algorithm learns patterns

**Inference** (fast, done many times): Use trained model to predict on new input

## 💡 Key Concept: Surrogate Model (Emulator)

A **surrogate** = fast approximation of an expensive computation.

- Full model: Run 5-minute simulation
- Surrogate: ML prediction in 0.01 seconds

**Key:** Surrogates are approximations. Use confidence to know when to trust them!

In [2]:
# Demonstrate ML concepts with a simple example
import random

# Simulate "telemetry history" — past runs with resource usage
# In reality, this comes from .scalable/runs/ telemetry files
historical_data = []
for i in range(50):
    num_nodes = random.randint(100, 2000)
    # True relationship: memory ≈ 0.01 * num_nodes + noise
    actual_memory_gb = 0.01 * num_nodes + random.gauss(0, 1)
    historical_data.append({
        "num_nodes": num_nodes,
        "memory_used_gb": round(max(1, actual_memory_gb), 1)
    })

print("Simulated telemetry history (50 past runs):")
print(f"  Sample: nodes={historical_data[0]['num_nodes']} → memory={historical_data[0]['memory_used_gb']}GB")
print(f"  Sample: nodes={historical_data[25]['num_nodes']} → memory={historical_data[25]['memory_used_gb']}GB")
print(f"  Sample: nodes={historical_data[49]['num_nodes']} → memory={historical_data[49]['memory_used_gb']}GB")
print(f"\n💡 Pattern: more nodes → more memory needed")

Simulated telemetry history (50 past runs):
  Sample: nodes=1688 → memory=16.5GB
  Sample: nodes=270 → memory=3.6GB
  Sample: nodes=988 → memory=8.5GB

💡 Pattern: more nodes → more memory needed


In [3]:
# Simple "ML model" — learn the pattern from data
# (In Scalable, this uses scikit-learn gradient boosting)

# Training: find the relationship
nodes = [d['num_nodes'] for d in historical_data]
memory = [d['memory_used_gb'] for d in historical_data]

# Simple linear fit (real Scalable uses gradient boosting)
avg_ratio = sum(m/n for n, m in zip(nodes, memory)) / len(nodes)

# Inference: predict for new inputs
new_scenario_nodes = 1500
predicted_memory = avg_ratio * new_scenario_nodes

print(f"ML Model trained on {len(historical_data)} past runs")
print(f"\nPrediction for new scenario (1500 nodes):")
print(f"  Predicted memory needed: {predicted_memory:.1f} GB")
print(f"  Recommended allocation: {int(predicted_memory * 1.2 + 2)} GB (with 20% headroom)")
print(f"\n💡 Without ML: guess '16GB' for everything (wasteful or insufficient)")
print(f"   With ML: data-driven recommendation specific to your workload")

ML Model trained on 50 past runs

Prediction for new scenario (1500 nodes):
  Predicted memory needed: 14.9 GB
  Recommended allocation: 19 GB (with 20% headroom)

💡 Without ML: guess '16GB' for everything (wasteful or insufficient)
   With ML: data-driven recommendation specific to your workload


## 💡 Key Concept: Uncertainty

**Uncertainty** = how confident the model is in its prediction.

```
Prediction: memory = 12GB ± 3GB (confidence: 87%)
```

- **High confidence** → trust the prediction, use the fast emulator
- **Low confidence** → don't trust, use the full (expensive) model

This is **confidence-gated routing**: Scalable automatically chooses
fast vs. slow path based on how trustworthy the approximation is.

## 💡 Key Concept: Active Learning

**Active learning** = intelligently choosing which data to learn from next.

Instead of randomly running 1000 scenarios, ask:
"Where is the model LEAST confident?" → run those specific scenarios.

Result: ~150 full model runs instead of 1000, same accuracy!

In [4]:
# Demonstrate the emulator concept
import time

def full_simulation(scenario_id: int) -> float:
    """The expensive full model (2 seconds)."""
    time.sleep(2)
    return scenario_id * 42.5 + 100

def emulator_prediction(scenario_id: int) -> tuple:
    """The fast ML approximation (instant).
    Returns (prediction, confidence)."""
    # Simulated: good predictions for known range, uncertain outside
    prediction = scenario_id * 42.3 + 101  # Close but not exact
    confidence = 0.95 if scenario_id < 100 else 0.6  # Less confident for large IDs
    return prediction, confidence

# Confidence-gated routing
CONFIDENCE_THRESHOLD = 0.9

for scenario_id in [10, 50, 150]:
    pred, conf = emulator_prediction(scenario_id)
    
    if conf >= CONFIDENCE_THRESHOLD:
        print(f"Scenario {scenario_id}: Emulator (confidence={conf:.0%}) → {pred:.1f} [FAST]")
    else:
        print(f"Scenario {scenario_id}: Low confidence ({conf:.0%}) → running full model... ", end="")
        # In reality this would call full_simulation (slow)
        print(f"[would take 2s]")

print(f"\n💡 Threshold={CONFIDENCE_THRESHOLD:.0%}: use emulator when confident, full model otherwise")

Scenario 10: Emulator (confidence=95%) → 524.0 [FAST]
Scenario 50: Emulator (confidence=95%) → 2216.0 [FAST]
Scenario 150: Low confidence (60%) → running full model... [would take 2s]

💡 Threshold=90%: use emulator when confident, full model otherwise


## 📖 Vocabulary Summary

| Term | Definition |
|------|------------|
| Machine Learning | Finding patterns in data to make predictions |
| Training | Learning phase (slow, done once) |
| Inference | Prediction phase (fast, done many times) |
| Features | Input variables the model uses |
| Model | Mathematical function learned from data |
| Surrogate/Emulator | Fast approximation of expensive computation |
| Uncertainty | How confident the model is |
| Confidence Threshold | Minimum confidence to use fast path |
| Active Learning | Strategically choosing what to learn next |
| Cross-Validation | Testing model quality on held-out data |

In [5]:
# Cleanup
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print(f"Cleaned up {project_dir}")

Cleaned up /var/folders/8c/787g_pzs7wq7hljnnx3prdzr0000gn/T/scalable-beginner-09-rhfzmo8w
